# Institutional Convergence as I-POMDP

**The Setup:** Two agents working on a project together. Each agent has a hidden **role preference** parameter that determines whether they naturally prefer to lead or follow. Through repeated interactions, they need to establish a stable leader-follower relationship.

**The I-POMDP Framework:**
- **Hidden types**: Role preference θ (discrete levels)
- **Actions**: Assertiveness levels from DEFER_STRONG to ASSERT_STRONG
- **Observations**: Agents observe each other's actions and update beliefs about types
- **Sequential dynamics**: Over multiple rounds, agents learn and converge to stable institutions

**The Goal:** Show how stable leader-follower institutions emerge through belief updating and strategic action selection in a multi-round I-POMDP.

In [1]:
from memo import memo, domain
import jax
import jax.numpy as np
import matplotlib.pyplot as plt

In [2]:
# Action space: 4 levels of assertiveness
A = np.arange(4)  # actions
Assertiveness = np.array([0.0, 0.3, 0.7, 1.0])  # DEFER_STRONG, DEFER_WEAK, ASSERT_WEAK, ASSERT_STRONG

# Type space: role preferences
# Low theta = prefers to lead, High theta = prefers to follow
T = np.array([0.2, 0.5, 0.8])  # role preference types

# History domain - track 3 rounds of interaction
H = domain(
    a1=len(A), b1=len(A),  # round 1: alice, bob actions
    a2=len(A), b2=len(A),  # round 2: alice, bob actions
    a3=len(A), b3=len(A)   # round 3: alice, bob actions
)

print(f'Action space: {len(A)} actions')
print(f'Assertiveness levels: {Assertiveness}')
print(f'Type space: {len(T)} types')
print(f'Type values: {T}')
print(f'History space: {H._size} histories')

Action space: 4 actions
Assertiveness levels: [0.  0.3 0.7 1. ]
Type space: 3 types
Type values: [0.2 0.5 0.8]


AttributeError: 'domain' object has no attribute '_size'

In [ ]:
@jax.jit
def coordination_payoff(a_alice, a_bob):
    """Payoff for coordination quality.
    
    Best: one asserts strongly, other defers strongly (clear roles)
    Good: asymmetric actions (some role differentiation)
    Bad: both assert strongly (conflict) or both defer strongly (indecision)
    """
    alice_assert = Assertiveness[a_alice]
    bob_assert = Assertiveness[a_bob]
    
    # Measure role differentiation (want asymmetry)
    diff = np.abs(alice_assert - bob_assert)
    
    # Measure average assertiveness (want someone to lead)
    avg = (alice_assert + bob_assert) / 2.0
    
    # Reward differentiation and having at least one leader
    # Penalize both being too passive or too aggressive
    base_payoff = diff * avg
    
    # Penalty for conflict (both assert strongly)
    conflict_penalty = np.minimum(alice_assert, bob_assert) * 0.5
    
    # Penalty for indecision (both defer strongly)
    indecision_penalty = np.maximum(0, 0.5 - avg)
    
    return base_payoff - conflict_penalty - indecision_penalty

@jax.jit
def role_fit_utility(action, theta):
    """Intrinsic utility from acting according to role preference.
    
    theta ≈ 0: prefers to lead (high assertiveness)
    theta ≈ 1: prefers to follow (low assertiveness)
    """
    assertiveness = Assertiveness[action]
    # Utility is higher when assertiveness matches (1 - theta)
    ideal_assertiveness = 1.0 - theta
    return -np.abs(assertiveness - ideal_assertiveness)

@jax.jit
def total_payoff_alice(a_alice, a_bob, theta_alice):
    """Alice's total payoff: coordination + role fit"""
    coord = coordination_payoff(a_alice, a_bob)
    fit = role_fit_utility(a_alice, theta_alice)
    return coord + 0.3 * fit  # weight role fit less than coordination

@jax.jit
def total_payoff_bob(a_alice, a_bob, theta_bob):
    """Bob's total payoff: coordination + role fit"""
    coord = coordination_payoff(a_alice, a_bob)
    fit = role_fit_utility(a_bob, theta_bob)
    return coord + 0.3 * fit

# Test payoffs
print("Sample payoffs:")
print(f"Both defer strongly: {coordination_payoff(0, 0):.3f}")
print(f"Both assert strongly: {coordination_payoff(3, 3):.3f}")
print(f"Alice asserts strong, Bob defers strong: {coordination_payoff(3, 0):.3f}")
print(f"Alice asserts weak, Bob defers weak: {coordination_payoff(2, 1):.3f}")

In [ ]:
@jax.jit
def is_init(h):
    """Check if history is initial (all zeros)"""
    return h == 0

@jax.jit
def Tr(r, h, a_alice, a_bob, h_):
    """Transition: update history h to h_ with actions (a_alice, a_bob) at round r"""
    z = H._tuple(h)
    z = np.array(z)
    z = z.at[r * 2].set(a_alice)
    z = z.at[r * 2 + 1].set(a_bob)
    return h_ == H(*z)

print("Dynamics functions defined")

In [ ]:
@memo(cache=True)
def hist[ta: T, tb: T, h: H](r, level, beta):
    """Prior over histories h at start of round r, conditioned on types ta, tb.
    
    This represents the agents' shared knowledge of how the game has unfolded
    given their types and rational play.
    """
    world: knows(ta, tb)
    # Start with history at round r-1
    world: chooses(h in H, wpp=hist[ta, tb, h](r - 1, level, beta) if r > 1 else is_init(h))
    # Alice and Bob make moves
    world: chooses(a_alice in A, wpp=exp(beta * alice[h, ta, a_alice](r - 1, level, beta)))
    world: chooses(a_bob in A, wpp=exp(beta * bob[h, tb, a_alice, a_bob](r - 1, level, beta)))
    # History gets updated
    world: chooses(h_ in H, wpp=Tr(r - 1, h, a_alice, a_bob, h_))
    return Pr[world.h_ == h]

@memo(cache=True)
def alice[h: H, ta: T, a_alice: A](r, level, beta):
    """Q-function for Alice conditioned on history h, own type ta.
    
    Alice:
    - Knows her own type ta
    - Has uncertainty over Bob's type tb
    - Updates beliefs about tb based on observed history h
    - Chooses action to maximize expected utility looking ahead
    """
    alice: knows(ta)
    alice: thinks[
        bob: knows(ta),
        bob: chooses(tb in T, wpp=1),  # Alice's prior over Bob's type
        bob: chooses(h in H, wpp=(hist[ta, tb, h](r, level - 1, beta) if r > 0 and level > 0 else is_init(h)) + 1e-3)
    ]
    alice: observes [bob.h] is h  # Alice updates beliefs about bob.tb given observed history
    alice: knows(a_alice)
    return alice[
        imagine[  # Q-value for choosing a_alice
            bob: knows(a_alice),
            bob: chooses(a_bob in A, wpp=exp(beta * bob[h, tb, a_alice, a_bob](r, level - 1, beta)) if level > 0 else 1),
            bob: chooses(h_ in H, wpp=Tr(r, h, a_alice, a_bob, h_)),
            bob: chooses(a_alice_ in A, wpp=exp(beta * alice[h_, ta, a_alice_](r + 1, level, beta)) if r < 3 and level > 0 else 1),
            E[
                total_payoff_alice(a_alice, bob.a_bob, ta)
                + (alice[bob.h_, ta, bob.a_alice_](r + 1, level, beta) if r < 3 and level > 0 else 0)
            ]
        ]
    ]

@memo(cache=True)
def bob[h: H, tb: T, a_alice: A, a_bob: A](r, level, beta):
    """Q-function for Bob conditioned on history h, own type tb, Alice's current action.
    
    Bob:
    - Knows his own type tb
    - Has uncertainty over Alice's type ta
    - Updates beliefs about ta based on observed history h and current action a_alice
    - Chooses action to maximize expected utility looking ahead
    """
    bob: knows(tb)
    bob: thinks[
        alice: knows(tb),
        alice: chooses(ta in T, wpp=1),  # Bob's prior over Alice's type
        alice: chooses(h in H, wpp=(hist[ta, tb, h](r, level - 1, beta) if r > 0 and level > 0 else is_init(h)) + 1e-3),
        alice: chooses(a_alice in A, wpp=exp(beta * alice[h, ta, a_alice](r, level - 1, beta)) if level > 0 else 1)
    ]
    bob: observes [alice.h] is h
    bob: observes [alice.a_alice] is a_alice  # Bob observes Alice's action this round
    bob: knows(a_bob)
    return bob[
        imagine[  # Q-value for choosing a_bob
            alice: knows(a_bob),
            alice: chooses(h_ in H, wpp=Tr(r, h, a_alice, a_bob, h_)),
            alice: chooses(a_alice_ in A, wpp=exp(beta * alice[h_, ta, a_alice_](r + 1, level, beta)) if r < 3 and level > 0 else 1),
            alice: chooses(a_bob_ in A, wpp=exp(beta * bob[h_, tb, a_alice_, a_bob_](r + 1, level, beta)) if r < 3 and level > 0 else 1),
            E[
                total_payoff_bob(alice.a_alice, a_bob, tb)
                + (bob[alice.h_, tb, alice.a_alice_, alice.a_bob_](r + 1, level, beta) if r < 3 and level > 0 else 0)
            ]
        ]
    ]

print("I-POMDP model defined")

In [ ]:
%%time

beta = 5.0
level = 3

# Solve Alice's policy at round 0 (initial)
alice_q = alice(r=0, level=level, beta=beta)
alice_policy = np.exp(beta * alice_q) / np.exp(beta * alice_q).sum(axis=-1, keepdims=True)

h = 0  # initial history
action_names = ['DEFER_STRONG', 'DEFER_WEAK', 'ASSERT_WEAK', 'ASSERT_STRONG']

print('Alice\'s policy at Round 1 (initial):")
for ti, theta in enumerate(T):
    print(f'\n  Type θ={theta:.1f} ({"\'leader\' preference" if theta < 0.5 else "flexible" if theta == 0.5 else "\'follower\' preference"}):")
    for ai, action in enumerate(action_names):
        prob = alice_policy[h, ti, ai]
        if prob > 0.01:  # Only show non-negligible probabilities
            print(f'    {action:15s} (assert={Assertiveness[ai]:.1f}): {prob:.3f}')

In [ ]:
%%time

# Solve Bob's policy at round 0 (initial)
bob_q = bob(r=0, level=level, beta=beta)
bob_policy = np.exp(beta * bob_q) / np.exp(beta * bob_q).sum(axis=-1, keepdims=True)

h = 0  # initial history

print('Bob\'s policy at Round 1 (initial):')
print('\nAfter Alice chooses DEFER_STRONG:')
for ti, theta in enumerate(T):
    print(f'\n  Type θ={theta:.1f}:')
    for ai, action in enumerate(action_names):
        prob = bob_policy[h, ti, 0, ai]  # 0 = Alice's DEFER_STRONG
        if prob > 0.01:
            print(f'    {action:15s} (assert={Assertiveness[ai]:.1f}): {prob:.3f}')

print('\n\nAfter Alice chooses ASSERT_STRONG:')
for ti, theta in enumerate(T):
    print(f'\n  Type θ={theta:.1f}:')
    for ai, action in enumerate(action_names):
        prob = bob_policy[h, ti, 3, ai]  # 3 = Alice's ASSERT_STRONG
        if prob > 0.01:
            print(f'    {action:15s} (assert={Assertiveness[ai]:.1f}): {prob:.3f}')

In [ ]:
# Analyze how beliefs and actions evolve over rounds
# TODO: Sample trajectories and show how institutions emerge
print("Analysis of institutional convergence over multiple rounds...")
print("This would show how initial uncertainty resolves into stable leader-follower roles.")